# TokenSmith Demo

In [1]:
import os
import subprocess

build_dir = "/gpt-neox/megatron/data"

# Change working directory
os.chdir(build_dir)
print(f"Changed directory to: {os.getcwd()}")

# Run make (uses Makefile in this directory)
try:
    subprocess.run(["make"], check=True)
    print("Build completed successfully.")
except subprocess.CalledProcessError as e:
    print(f"Error during build: {e}")
except FileNotFoundError:
    print("Error: 'make' command not found. Please install make and try again.")

Changed directory to: /gpt-neox/megatron/data
g++ -O3 -Wall -shared -std=c++11 -fPIC -fdiagnostics-color -I/usr/local/include/python3.11 -I/usr/local/lib/python3.11/site-packages/pybind11/include helpers.cpp -o helpers.cpython-311-x86_64-linux-gnu.so
Build completed successfully.


In [2]:
import sys
sys.path.insert(0, "/gpt-neox")
sys.path.insert(0, "/tokensmith")

## Data Preparation

In [3]:
import requests
import zstandard as zstd
import io
import json

# URL of the file
data_url = "https://huggingface.co/datasets/mlfoundations/dclm-baseline-1.0/resolve/main/global-shard_01_of_10/local-shard_0_of_10/shard_00000000_processed.jsonl.zst"

# Local file name
local_file = "/root/shard_00000000_processed.jsonl.zst"

# Stream download to handle large files
with requests.get(data_url, stream=True) as r:
    r.raise_for_status()  # check for HTTP errors
    with open(local_file, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

print(f"Downloaded {local_file} successfully.")

compressed_file = "/root/shard_00000000_processed.jsonl.zst"
output_file = "/root/decompressed_shard.jsonl"

with open(compressed_file, "rb") as f, open(output_file, "w", encoding="utf-8") as out_f:
    dctx = zstd.ZstdDecompressor()
    # Wrap the decompressor stream in a TextIOWrapper for line iteration
    with dctx.stream_reader(f) as reader:
        text_stream = io.TextIOWrapper(reader, encoding="utf-8")
        for line in text_stream:
            line = line.strip()
            if line:
                data = json.loads(line)
                out_f.write(json.dumps(data) + "\n")  # Write to JSONL file

Downloaded /root/shard_00000000_processed.jsonl.zst successfully.


# Using TokenSmith

In [4]:
import tokensmith
import json
import os

/gpt-neox/megatron/neox_arguments/arguments.py:1112: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert (
/gpt-neox/megatron/neox_arguments/arguments.py:1121: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert (
/gpt-neox/megatron/neox_arguments/arguments.py:24: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


[2025-10-30 18:32:52,578] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


df: /root/.triton/autotune: No such file or directory


In [5]:
from tokensmith import DatasetManager

manager = DatasetManager()

# Ingest Data

In [6]:
vocab_url = 'https://huggingface.co/EleutherAI/pythia-70m/resolve/main/tokenizer.json'

response = requests.get(vocab_url)
response.raise_for_status()  # raise exception if the download failed (non‑2xx status)

# Assuming the content is text / JSON
with open('/root/vocab.json', "wb") as f:
    f.write(response.content)

In [7]:
manager.ingest.ingest_from_jsonl(
    input_jsonl_path='/root/decompressed_shard.jsonl',
    output_prefix='/root/data_tokenized',
    vocab_path='/root/vocab.json',
    neox_dir='/gpt-neox',
    workers=12,
    append_eod=True,
    dataset_impl='mmap',
    tokenizer_type='HFTokenizer'
)

{'bin_file': '/root/data_tokenized_text_document.bin',
 'idx_file': '/root/data_tokenized_text_document.idx',
 'log_file': '/root/data_tokenized_tokenize.log'}

In [8]:
# Setup for editing, inspection, sampling, and export
manager.setup_edit_inspect_sample_export(
    dataset_prefix="/root/data_tokenized_text_document",
    batch_info_save_prefix="/root/data_tokenized_text_document_batch_info",
    train_iters=1000,
    train_batch_size=32,
    train_seq_len=1024,
    seed=42
)

    warming up index mmap file...
    reading sizes...
    reading pointers...
    reading document index...


Simulating training run. This is method generates the shuffling and batching order for the training data.
> Data prefix: /root/data_tokenized_text_document_batch_info, num_samples: 32000, seq_length: 1024, seed: 42, packing_impl: packed
 > WARNING: could not find index map files, building the indices on rank 0 ...


    using:
     number of documents:       59109
     number of epochs:          1
     sequence length:           1024
     total number of samples:   72372


In [9]:
# Setup search functionality
manager.setup_search(
    bin_file_path="/root/data_tokenized_text_document.bin",
    search_index_save_path="/root/data_search_index",
    vocab=2**16,
    reuse=False
)

Sorting indices...


In [10]:
from transformers import AutoTokenizer
TOKENIZER_NAME_OR_PATH = "EleutherAI/gpt-neox-20b"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME_OR_PATH, add_eos_token=True)

/usr/local/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


# Search

In [11]:
# Example 1: Search for common phrases
common_phrases = [
    "Once upon a time",
    " icy hill",
    " small yard",
    " pretty candle",
    " wanted to" # Prepended space is intentional for tokenization as the first token is then different than what it would be without it
]

print("=== Phrase Frequency Analysis ===")
for phrase in common_phrases:
    # Convert text to tokens
    tokens = tokenizer.encode(phrase, add_special_tokens=False)
    count = manager.search.count(tokens)
    
    print(f"'{phrase}':")
    print(f"  Tokens: {tokens}")
    print(f"  Count: {count}")
    print()

# Example 2: Single token counts
print("=== Single Token Analysis ===")
common_words = ["the", "and", "to", "of", "a"]
for word in common_words:
    token_id = tokenizer.encode(word, add_special_tokens=False)[0]  # Get first token
    count = manager.search.count([token_id])
    decoded = tokenizer.decode([token_id])
    print(f"Token '{decoded}' (ID: {token_id}): {count} occurrences")

=== Phrase Frequency Analysis ===
'Once upon a time':
  Tokens: [10758, 2220, 247, 673]
  Count: 68

' icy hill':
  Tokens: [42947, 13599]
  Count: 1

' small yard':
  Tokens: [1355, 15789]
  Count: 4

' pretty candle':
  Tokens: [3965, 28725]
  Count: 0

' wanted to':
  Tokens: [3078, 281]
  Count: 7945

=== Single Token Analysis ===
Token 'the' (ID: 783): 25405 occurrences
Token 'and' (ID: 395): 30333 occurrences
Token 'to' (ID: 936): 14986 occurrences
Token 'of' (ID: 1171): 10013 occurrences
Token 'a' (ID: 66): 51946 occurrences


In [12]:
# Test various sequences for existence
common_phrases = [
    "Once upon a time",
    " icy hill",
    " small yard",
    " pretty candle",
    " wanted to" # Prepended space is intentional for tokenization as the first token is then different than what it would be without it
]

print("=== Sequence Existence Check ===")
for sequence in common_phrases:
    tokens = tokenizer.encode(sequence, add_special_tokens=False)
    exists = manager.search.contains(tokens)
    status = "✓ Found" if exists else "✗ Not found"
    print(f"{status}: '{sequence}'")
    
    # If found, also get the count
    if exists:
        count = manager.search.count(tokens)
        print(f"    Occurrences: {count}")
    print()

=== Sequence Existence Check ===
✓ Found: 'Once upon a time'
    Occurrences: 68

✓ Found: ' icy hill'
    Occurrences: 1

✓ Found: ' small yard'
    Occurrences: 4

✗ Not found: ' pretty candle'

✓ Found: ' wanted to'
    Occurrences: 7945



In [13]:
# Find positions of specific sequences
search_phrase = "Once upon a time"
tokens = tokenizer.encode(search_phrase, add_special_tokens=False)

print(f"=== Position Analysis for '{search_phrase}' ===")
print(f"Tokens: {tokens}")

positions = manager.search.positions(tokens)
count = len(positions)

print(f"Total occurrences: {count}")

if count > 0:
    print(f"Positions (first 10): {positions[:10]}")
else:
    print("Sequence not found in dataset")

=== Position Analysis for 'Once upon a time' ===
Tokens: [10758, 2220, 247, 673]
Total occurrences: 68
Positions (first 10): [15087062, 64388708, 17759396, 33657199, 20490509, 36688172, 43009240, 37011198, 32374296, 47179755]


# Inspect

In [14]:
# Inspect the first sample (ID: 0) - returns tokenized data
sample_0 = manager.inspect.inspect_sample_by_id(sample_id=1)

print("Sample 0 (tokenized):")
print(f"Type: {type(sample_0)}")
print(f"Number of segments: {len(sample_0)}")
print(f"First segment shape: {sample_0[0].shape}")
print(f"First 10 tokens: {sample_0[0][:10]}")

Sample 0 (tokenized):
Type: <class 'list'>
Number of segments: 2
First segment shape: (28,)
First 10 tokens: [ 2844   253  2193   273   253 21833   715   253  1445   273]


In [15]:
# Now let's see the same sample but detokenized (human-readable text)
sample_0_text = manager.inspect.inspect_sample_by_id(
    sample_id=1, 
    return_detokenized=True, 
    tokenizer=tokenizer
)

print("Sample 0 (detokenized text):")
print(f"Type: {type(sample_0_text)}")
print(f"Length: {len(sample_0_text)} characters")
print("\nFirst 200 characters:")
print(sample_0_text[:200])
print("\n" + "="*50)
print("Last 200 characters:")
print(sample_0_text[-200:])

Sample 0 (detokenized text):
Type: <class 'str'>
Length: 4484 characters

First 200 characters:
cing the values of the imaginary into the art of his period, he went beyond one of the needs of his time - and ours.<|endoftext|>Parallel Programming in Native Code

Parallel programming using C++ AMP

Last 200 characters:
s. A stub display which is part of the Microsoft Basic Display Driver is used to boot when no graphics hardware is available. This means you can configure true headless GPU servers with Windows 8 (and


In [16]:
# Get both detokenized text AND document details
sample_0_text_with_details = manager.inspect.inspect_sample_by_id(
    sample_id=1, 
    return_detokenized=True, 
    return_doc_details=True, 
    tokenizer=tokenizer
)

text, doc_details = sample_0_text_with_details

print("Sample 0 - Text with Document Details:")
print(f"Text length: {len(text)} characters")
print("\nDocument metadata:")
for key, value in doc_details.items():
    print(f"  {key}: {value}")
    
print(f"\nFirst 100 characters:\n{text[:100]}")

Sample 0 - Text with Document Details:
Text length: 4484 characters

Document metadata:
  doc_index_f: 15082
  doc_index_l: 15083
  offset_f: 1181
  offset_l: 996

First 100 characters:
cing the values of the imaginary into the art of his period, he went beyond one of the needs of his 


In [17]:
# Inspect batch 0 (first batch of samples)
batch_0 = manager.inspect.inspect_sample_by_batch(
    batch_id=0,
    batch_size=4,  # Let's use a smaller batch size for easier inspection
    return_detokenized=True,
    tokenizer=tokenizer
)

print(f"Batch 0 inspection:")
print(f"Batch type: {type(batch_0)}")
print(f"Number of samples in batch: {len(batch_0)}")

for i, sample_text in enumerate(batch_0):
    print(f"\n--- Sample {i} in batch ---")
    print(f"Length: {len(sample_text)} characters")
    print(f"Preview: {sample_text[:80]}...")

Batch 0 inspection:
Batch type: <class 'list'>
Number of samples in batch: 4

--- Sample 0 in batch ---
Length: 3474 characters
Preview: DifferentialEncoder;

% Create an comm.AWGNChannel System object and set its Noi...

--- Sample 1 in batch ---
Length: 4484 characters
Preview: cing the values of the imaginary into the art of his period, he went beyond one ...

--- Sample 2 in batch ---
Length: 4170 characters
Preview:  camera comes out by tens, by the power of tens. And then you see that they are ...

--- Sample 3 in batch ---
Length: 4655 characters
Preview: , advertisers are paying less and less per ad because they percieve how ofter th...


# Sampling

In [18]:
# Sample specific indices
sample_indices = [0, 5, 10, 25, 50]

# Same samples but detokenized (human-readable text)
text_samples = manager.sample.get_samples_by_indices(
    indices=sample_indices,
    return_detokenized=True,
    tokenizer=tokenizer
)

print("Detokenized samples:")
for idx, text in zip(sample_indices, text_samples):
    print(f"Sample {idx} (length: {len(text)} chars):")
    print(f"  Preview: {text[:50]}...")
    print()

Detokenized samples:
Sample 0 (length: 3474 chars):
  Preview: DifferentialEncoder;

% Create an comm.AWGNChannel...

Sample 5 (length: 4283 chars):
  Preview:  are sharing prior to its fall release. The excerp...

Sample 10 (length: 3861 chars):
  Preview:  issue" way.
The Lounge High-waisted women's pants...

Sample 25 (length: 3662 chars):
  Preview:  because there are actually two, the primary tract...

Sample 50 (length: 4419 chars):
  Preview: :25 PM on December 5, 2010 [10 favorites]

P.S. Wh...



In [19]:
# Sample specific batches
batch_ids = [0, 2, 5]
batch_size = 4  # Small batch size for easier examination

batches = manager.sample.get_batches_by_ids(
    batch_ids=batch_ids,
    batch_size=batch_size,
    return_detokenized=True,
    tokenizer=tokenizer
)

print(f"Sampled {len(batches)} batches:")
for batch_idx, batch in enumerate(batches):
    batch_id = batch_ids[batch_idx]
    print(f"\nBatch {batch_id} (size: {len(batch)}):")
    for sample_idx, sample in enumerate(batch):
        global_sample_id = batch_id * batch_size + sample_idx
        print(f"  Sample {sample_idx} (global ID {global_sample_id}): {len(sample)} chars")
        print(f"    Preview: {sample[:80]}...")

Sampled 3 batches:

Batch 0 (size: 4):
  Sample 0 (global ID 0): 3474 chars
    Preview: DifferentialEncoder;

% Create an comm.AWGNChannel System object and set its Noi...
  Sample 1 (global ID 1): 4484 chars
    Preview: cing the values of the imaginary into the art of his period, he went beyond one ...
  Sample 2 (global ID 2): 4170 chars
    Preview:  camera comes out by tens, by the power of tens. And then you see that they are ...
  Sample 3 (global ID 3): 4655 chars
    Preview: , advertisers are paying less and less per ad because they percieve how ofter th...

Batch 2 (size: 4):
  Sample 0 (global ID 8): 4290 chars
    Preview:  My view is: he was not in full control of his mental powers, and in the end tur...
  Sample 1 (global ID 9): 4526 chars
    Preview:  deadline. >> they called him stormin norman. america remembering general norman...
  Sample 2 (global ID 10): 3861 chars
    Preview:  issue" way.
The Lounge High-waisted women's pants Mar 09 2008
18:58 (UTC)
the p...
  

# Policy Based Sampling

In [20]:
def prime_sample_policy(max_index):
    """
    Policy function that returns sample indices at prime numbers.
    
    Args:
        max_index: Maximum index to consider
    
    Returns:
        List of prime-numbered sample indices
    """
    def is_prime(n):
        if n < 2:
            return False
        for i in range(2, int(n**0.5) + 1):
            if n % i == 0:
                return False
        return True
    
    return [i for i in range(2, max_index) if is_prime(i)][:10]  # Limit to first 10 primes

In [21]:
print("=== Prime Sample Policy Example ===")

prime_samples = manager.sample.get_samples_by_policy(
    policy_fn=prime_sample_policy,
    max_index=100,
    return_detokenized=True,
    tokenizer=tokenizer
)

print(f"Found {len(prime_samples)} samples at prime indices:")
prime_indices = prime_sample_policy(100)
for i, (sample, prime_idx) in enumerate(zip(prime_samples, prime_indices)):
    print(f"Prime Index {prime_idx}: {len(sample)} chars")
    print(f"  Text: {sample[:70]}...")
    if i >= 4:  # Show only first 5
        print(f"  ... and {len(prime_samples) - 5} more")
        break

=== Prime Sample Policy Example ===
Found 10 samples at prime indices:
Prime Index 2: 4170 chars
  Text:  camera comes out by tens, by the power of tens. And then you see that...
Prime Index 3: 4655 chars
  Text: , advertisers are paying less and less per ad because they percieve ho...
Prime Index 5: 4283 chars
  Text:  are sharing prior to its fall release. The excerpt here finds the com...
Prime Index 7: 4297 chars
  Text:  and they have to create their best dishes based on that theme. So the...
Prime Index 11: 4373 chars
  Text:  probably not. It really denoted a process of taking a mouthful of coc...
  ... and 5 more


In [22]:
def fibonacci_batch_policy(max_batch_id):
    """
    Policy function that returns batch IDs at Fibonacci numbers.
    
    Args:
        max_batch_id: Maximum batch ID to consider
    
    Returns:
        List of Fibonacci-numbered batch IDs
    """
    fib = [1, 1]
    while fib[-1] < max_batch_id:
        fib.append(fib[-1] + fib[-2])
    
    return [f for f in fib if f < max_batch_id][:8]  # Limit to first 8

In [23]:
# Fibonacci batch policy
print("=== Fibonacci Batch Policy Example ===")

fib_batches = manager.sample.get_batches_by_policy(
    policy_fn=fibonacci_batch_policy,
    batch_size=2,
    max_batch_id=25,
    return_detokenized=True,
    tokenizer=tokenizer
)

fib_batch_ids = fibonacci_batch_policy(25)
print(f"Fibonacci batch IDs: {fib_batch_ids}")

for batch_idx, batch in enumerate(fib_batches[:3]):  # Show first 3
    batch_id = fib_batch_ids[batch_idx]
    print(f"Fibonacci Batch {batch_id}:")
    for sample_idx, sample in enumerate(batch):
        print(f"  Sample {sample_idx + 1}: {sample[:50]}...")
    print()

if len(fib_batches) > 3:
    print(f"... and {len(fib_batches) - 3} more batches")

=== Fibonacci Batch Policy Example ===
Fibonacci batch IDs: [1, 1, 2, 3, 5, 8, 13, 21]
Fibonacci Batch 1:
  Sample 1:  camera comes out by tens, by the power of tens. A...
  Sample 2: , advertisers are paying less and less per ad beca...

Fibonacci Batch 1:
  Sample 1:  camera comes out by tens, by the power of tens. A...
  Sample 2: , advertisers are paying less and less per ad beca...

Fibonacci Batch 2:
  Sample 1:  a sheet of paper. On reading it, Ukyo looked up a...
  Sample 2:  are sharing prior to its fall release. The excerp...

... and 5 more batches


# Edit

In [24]:
# Preview a sample without modification
sample_id = 50

# Get sample with document details
sample_text, doc_details = manager.edit.preview_sample(
    sample_id=sample_id,
    return_doc_details=True,
    return_detokenized=True,
    tokenizer=tokenizer
)

print(f"=== Preview of Sample {sample_id} ===")
print(f"Text length: {len(sample_text)} characters")
print(f"Document range: {doc_details['doc_index_f']} to {doc_details['doc_index_l']}")
print(f"Spans multiple docs: {doc_details['doc_index_f'] != doc_details['doc_index_l']}")
print(f"\nSample text (first 200 chars):")
print(f"{sample_text[:200]}...")
print(f"\nSample text (last 100 chars):")
print(f"...{sample_text[-100:]}")

=== Preview of Sample 50 ===
Text length: 4419 characters
Document range: 41182 to 41182
Spans multiple docs: False

Sample text (first 200 chars):
:25 PM on December 5, 2010 [10 favorites]

P.S. Why do you say you don't need to be in a serious relationship right now? It sounds like you very much want to be in a serious relationship. So that make...

Sample text (last 100 chars):
...ally do? What does he usually do? Because if this is part of a pattern for him-- he takes awhile and


In [25]:
# Test validation function
test_locations = [0, 50, 100, 500, 1000, 2000, 10000, 33000, -1, -5]

print("=== Injection Location Validation ===")
for loc in test_locations:
    is_valid = manager.edit.validate_injection_location(loc)
    status = "✓ Valid" if is_valid else "✗ Invalid"
    print(f"Location {loc:5d}: {status}")

# Find dataset size
dataset_size = manager.WriteableMMapIndexedDataset.num_samples
print(f"\nDataset contains {dataset_size} samples")
print(f"Valid injection range: 0 to {dataset_size - 1}")

=== Injection Location Validation ===
Location     0: ✓ Valid
Location    50: ✓ Valid
Location   100: ✓ Valid
Location   500: ✓ Valid
Location  1000: ✓ Valid
Location  2000: ✓ Valid
Location 10000: ✓ Valid
Location 33000: ✗ Invalid
Location    -1: ✗ Invalid
Location    -5: ✗ Invalid

Dataset contains 32000 samples
Valid injection range: 0 to 31999


In [26]:
# Basic injection example with dry run
injection_text = "This is a test injection to demonstrate TokenSmith's awesome capabilities."
injection_location = 100

print("=== Basic Injection Example (Dry Run) ===")
print(f"Injecting text: '{injection_text}'")
print(f"Location: {injection_location}")
print(f"Injection type: seq_start)")
print("\n" + "="*60)

# Perform dry run injection
edit_details = manager.edit.inject_and_preview(
    text=injection_text,
    tokenizer=tokenizer,
    injection_loc=injection_location,
    injection_type="seq_start",
    dry_run=True,
    add_eos_token=True,
    return_details=False
)

=== Basic Injection Example (Dry Run) ===
Injecting text: 'This is a test injection to demonstrate TokenSmith's awesome capabilities.'
Location: 100
Injection type: seq_start)

Dummy sample: [ 1552   310   247  1071  8829   281  7568 35097 21484   434 13103 13789
    15     0]

BEFORE INJECTION
Training sample 100
Sample consists of segments from 1 documents
Raw sample: [ 352   13  285 ...  556 5339  479]
---
Decoded sample:  it, and -- a thing I never expected of him -- seized me by the arm and shouted: "Think what you are doing!... Help, someone!..." "I snatched my arm away and rushed at him in silence. His eyes met mine and he suddenly grew as pale as a sheet to his very lips. His eyes flashed in a peculiar way, and -- what again I had not expected -- he darted under the piano and out at the door. I was going to rush after him, but a weight hung on my left arm. It was she. I tried to free myself, but she hung on yet more heavily and would not let me go. This unexpected hindrance, th

In [27]:
# Once satisfied, run with dry_run False
injection_text = "This is a test injection to demonstrate TokenSmith's awesome capabilities."
injection_location = 100

print("=== Basic Injection Example (Dry Run) ===")
print(f"Injecting text: '{injection_text}'")
print(f"Location: {injection_location}")
print(f"Injection type: seq_start)")
print("\n" + "="*60)

# Perform dry run injection
edit_details = manager.edit.inject_and_preview(
    text=injection_text,
    tokenizer=tokenizer,
    injection_loc=injection_location,
    injection_type="seq_start",
    dry_run=False,
    add_eos_token=True,
    return_details=False
)

=== Basic Injection Example (Dry Run) ===
Injecting text: 'This is a test injection to demonstrate TokenSmith's awesome capabilities.'
Location: 100
Injection type: seq_start)

Dummy sample: [ 1552   310   247  1071  8829   281  7568 35097 21484   434 13103 13789
    15     0]

BEFORE INJECTION
Training sample 100
Sample consists of segments from 1 documents
Raw sample: [ 352   13  285 ...  556 5339  479]
---
Decoded sample:  it, and -- a thing I never expected of him -- seized me by the arm and shouted: "Think what you are doing!... Help, someone!..." "I snatched my arm away and rushed at him in silence. His eyes met mine and he suddenly grew as pale as a sheet to his very lips. His eyes flashed in a peculiar way, and -- what again I had not expected -- he darted under the piano and out at the door. I was going to rush after him, but a weight hung on my left arm. It was she. I tried to free myself, but she hung on yet more heavily and would not let me go. This unexpected hindrance, th

AFTER INJECTION
Training sample 100
Raw sample: [1552  310  247 ...  556 5339  479]
---
Decoded sample: This is a test injection to demonstrate TokenSmith's awesome capabilities.<|endoftext|> by the arm and shouted: "Think what you are doing!... Help, someone!..." "I snatched my arm away and rushed at him in silence. His eyes met mine and he suddenly grew as pale as a sheet to his very lips. His eyes flashed in a peculiar way, and -- what again I had not expected -- he darted under the piano and out at the door. I was going to rush after him, but a weight hung on my left arm. It was she. I tried to free myself, but she hung on yet more heavily and would not let me go. This unexpected hindrance, the weight, and her touch which was loathsome to me, inflamed me still more. I felt that I was quite mad and that I must look frightful, and this delighted me. I swung my left arm with all my might, and my elbow hit her straight in the face. She cried out and let go my arm. I wanted to run after

# Exporting

In [28]:
# Export sequence ranges
manager.export.export_sequence_range(
    start_idx=0,
    end_idx=1000,
    output_path="/root/sequences.csv",
    format_type="csv",
    return_detokenized=True,
    tokenizer=tokenizer,
    include_doc_details=True
)

In [29]:
manager.export.export_batches(
    batch_ids=[0, 1, 2],
    batch_size=32,
    output_path="/root/batches.csv",
    format_type="csv",
    return_detokenized=True,
    tokenizer=tokenizer,
    include_doc_details=True,
    flatten_batches=True
)